In [0]:
source_dir = "/Volumes/my_catalog/my_volume/my_file_volume/source_erp/"
schema_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_schemas/px_cat_g1v2_bronze/"
checkpoint_dir = "/Volumes/my_catalog/my_volume/my_file_volume/_checkpoints/px_cat_g1v2_bronze/"
target_table = "my_catalog.bronze.PX_CAT_G1V2"

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_dir)      
      .option("header", "true")
      .option("pathGlobFilter", "PX_CAT_G1V2.csv")          # ✅ only this file
      .load(source_dir)
)

(df.writeStream
 .format("delta")
 .option("checkpointLocation", checkpoint_dir)
 .outputMode("append")
 .option("mergeSchema", "true")                            # allow schema evolution
 .trigger(once=True)                                       # run once then stop
 .toTable(target_table)
)